# Qwen2.5-1.5B QLoRA on Kaggle

Before running:
- Turn `Accelerator` on and select `GPU`
- Turn `Internet` on so the notebook can clone the GitHub repo and download the model
- Add your Kaggle dataset that contains `train_creative_train.jsonl` and `train_creative_valid.jsonl`


In [ ]:
DATASET_SLUG = 'vietnamese-poetry-data'
REPO_URL = 'https://github.com/Quangduong2703/vietnamese-poetry-ft.git'
REPO_DIR = '/kaggle/working/vietnamese-poetry-ft'
OUTPUT_DIR = '/kaggle/working/outputs/qwen25_15b_poetry_run1'
TRAIN_FILE = f'/kaggle/input/{DATASET_SLUG}/train_creative_train.jsonl'
VALID_FILE = f'/kaggle/input/{DATASET_SLUG}/train_creative_valid.jsonl'

print(TRAIN_FILE)
print(VALID_FILE)
print(OUTPUT_DIR)

In [ ]:
!nvidia-smi

In [ ]:
import torch
NPROC = torch.cuda.device_count()
print('Detected GPUs:', NPROC)
if NPROC < 1:
    raise RuntimeError('No GPU detected. Enable GPU in Notebook settings.')

In [ ]:
!git clone {REPO_URL} {REPO_DIR} || true
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -r /kaggle/working/vietnamese-poetry-ft/Source/requirements_qwen25_qlora.txt

In [ ]:
import json
from pathlib import Path

config_path = Path('/kaggle/working/vietnamese-poetry-ft/Source/configs/qwen25_15b_qlora_kaggle.json')
config = json.loads(config_path.read_text())
config['train_file'] = TRAIN_FILE
config['valid_file'] = VALID_FILE
config['output_dir'] = OUTPUT_DIR
config_path.write_text(json.dumps(config, indent=2))
print(config_path.read_text())

In [ ]:
!torchrun --standalone --nproc_per_node={NPROC} /kaggle/working/vietnamese-poetry-ft/Source/scripts/train_qwen25_15b_qlora.py \
  --config /kaggle/working/vietnamese-poetry-ft/Source/configs/qwen25_15b_qlora_kaggle.json

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/outputs/qwen25_15b_poetry_run1/runs

In [ ]:
!ls -R /kaggle/working/outputs/qwen25_15b_poetry_run1 | head -n 200